# Grad-CAM Visualization Toolkit

A configurable notebook for generating class-specific Grad-CAM heatmaps, image overlays, and an Excel workbook of attention-related metrics for **Torchvision DeepLabV3 with ResNet-50 or ResNet-101 backbones**.

## Features

- Processes an image list against one or more `.pth` checkpoints.
- Produces a heatmap PNG and a blended overlay PNG for each image/model/class combination where the model predicts that class.
- Fuses Grad-CAM maps from `layer2`, `layer3`, and `layer4` (each layer is normalized before fusion).
- Exports raw metrics, location/class summary statistics, and one-way ANOVA results to Excel.
- Records per-image/model errors in the workbook and writes a list of images that could not be found.
- Uses configurable paths and generic class labels instead of machine-specific paths or project-specific dataset names.

## Repository layout

Arrange your files like this, or update the configuration in the notebook to match your layout:

```text
project/
├── README.md
├── gradcam_visualization_toolkit.ipynb
├── image_filenames.txt
├── models/
│   ├── example_resnet50.pth
│   └── example_resnet101.pth
├── data/
│   ├── train/
│   ├── valid/
│   └── test/
└── outputs/
```

`image_filenames.txt` should contain one image filename per line. The image files are searched for recursively inside the configured source directories. If the same basename exists in more than one source directory, the first match is used.

## Requirements

Use Python 3.10 or newer. Install the dependencies in your environment (or run the optional install cell in the notebook):

```bash
pip install torch torchvision pandas pillow matplotlib scipy openpyxl numpy
```

Install a PyTorch/Torchvision combination appropriate for your CPU or CUDA environment. The package command above does not select a CUDA-specific PyTorch wheel for you.

## Configure before running

Open `gradcam_visualization_toolkit.ipynb` and update the **CONFIGURATION** section near the top of the code:

- `PROJECT_ROOT`: root of your project/repository.
- `MODEL_DIR`: directory containing checkpoint files.
- `IMAGE_LIST_FILE`: text file containing image basenames, one per line.
- `SOURCE_IMAGE_DIRS`: directories searched for source images.
- `OUTPUT_DIR`: destination for Excel and PNG output.
- `MODEL_FILES`: set to `None` to process every file matching `MODEL_GLOB`, or provide a list of checkpoint filenames.
- `NUM_CLASSES`: total model output classes, **including background class 0**.
- `CLASS_NAMES`: map foreground class IDs to readable folder/report labels. The example uses generic labels; replace them with the labels used during training.
- `INPUT_SIZE`: `(width, height)` used to resize images before inference. The default is `(1024, 768)`. Set it to `None` to preserve each image's original dimensions if that matches your model's training/evaluation setup.
- `DEVICE_REQUEST`: `auto`, `cpu`, `cuda`, or a specific device such as `cuda:0`.
- `LOCATION_KEYWORDS`: optional mapping used to assign location/region labels from filenames. Leave it empty if no location grouping is required; records will be labelled `Unknown` and location-based ANOVA will not have multiple groups to compare.

The filename of each checkpoint must contain `resnet50` or `resnet101`, because the loader uses that string to choose the Torchvision architecture. Checkpoints may be a raw state dictionary or a dictionary containing `model_state_dict` or `state_dict`. The architecture, number of classes, and classifier shapes must be compatible with the checkpoint trained for your task. Warnings about missing/unexpected keys require review; `strict=False` is used to make those mismatches visible rather than immediately stopping loading.

## Outputs

The notebook writes files under `OUTPUT_DIR`:

```text
outputs/gradcam_visualizations/
├── gradcam_metrics.xlsx
├── missing_images.txt             # only if listed images are not found
└── png/
    └── <model_name>/
        └── <class_name>/
            ├── <image_stem>__heatmap.png
            └── <image_stem>__overlay.png
```

The Excel workbook contains:

- `Raw_Data`: one row for each successfully computed image/model/class combination.
- `Location_Class_Summary`: mean, standard deviation, and count grouped by location and class.
- `ANOVA_Analysis`: one-way ANOVA results for metrics with at least two location groups containing more than two observations each.
- `Errors`: created only if any image/model processing errors were recorded.

A class only gets a heatmap/overlay when the model predicts at least one pixel of that class in the image. Therefore, the number of PNG files can be lower than `number of images × number of models × number of classes × 2`.

## Metric definitions

Metrics use the model's **predicted mask for the target class**, not a ground-truth mask:

- **Focus Ratio**: sum of fused CAM values inside the predicted class mask divided by the sum over the whole CAM.
- **Mean CAM Inside / Outside**: average fused CAM value inside or outside that predicted mask.
- **Entropy**: entropy of the CAM values after normalization to a distribution, using the natural logarithm.
- **Centroid Distance**: Euclidean distance between the CAM-weighted centroid inside the predicted mask and the geometric centroid of that mask, in pixels at the configured input resolution.

These definitions follow the fused layer2/layer3/layer4 metric workflow. They should not be assumed numerically identical to results from a different target layer, resize policy, class mapping, or entropy logarithm.

## Run

Run the notebook cells in order. The final cell calls `run_analysis()`. The process may be computationally expensive for many images and checkpoints; a CUDA-capable environment is recommended for larger jobs.

## Limitations

- This loader is designed for Torchvision DeepLabV3 with ResNet-50/101 backbones; it is not a universal Grad-CAM loader for arbitrary model families.
- The notebook does not calculate metrics against ground-truth segmentation masks.
- Verify class IDs, preprocessing, resize dimensions, checkpoint architecture, and class labels against your own training pipeline before interpreting the metrics.


## 1. Install dependencies (optional)

Run this cell only if the required packages are not already installed in the active environment.

In [ ]:
# Uncomment if needed, then run this cell:
# %pip install torch torchvision pandas pillow matplotlib scipy openpyxl numpy

## 2. Imports and configuration

Review and edit the paths, model files, class labels, image size, and device options in this cell before running the analysis.

In [ ]:
#!/usr/bin/env python3
"""Configurable Grad-CAM visualization + metrics for torchvision DeepLabV3 ResNet50/101."""
from pathlib import Path
import gc
import warnings

import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.models as tv_models
from PIL import Image

# =============================================================================
# CONFIGURATION — edit these values for your dataset/repository
# =============================================================================
PROJECT_ROOT = Path('.').resolve()
MODEL_DIR = PROJECT_ROOT / 'models'
IMAGE_LIST_FILE = PROJECT_ROOT / 'image_filenames.txt'  # one image filename per line
SOURCE_IMAGE_DIRS = [
    PROJECT_ROOT / 'data' / 'train',
    PROJECT_ROOT / 'data' / 'valid',
    PROJECT_ROOT / 'data' / 'test',
]
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'gradcam_visualizations'

# None = process every *.pth file in MODEL_DIR. Or specify e.g. ['model_a.pth'].
MODEL_FILES = None
MODEL_GLOB = '*.pth'

# Semantic-segmentation convention: class 0 is background; foreground classes start at 1.
NUM_CLASSES = 4
CLASS_NAMES = {
    1: 'class_1',
    2: 'class_2',
    3: 'class_3',
}

# Input size in (width, height). Set to None to retain each image's native size.
INPUT_SIZE = (1024, 768)
OVERLAY_ALPHA = 0.42
HEATMAP_COLORMAP = 'jet'
DEVICE_REQUEST = 'auto'  # 'auto', 'cpu', 'cuda', or 'cuda:0'
EXPECTED_IMAGE_COUNT = None  # e.g. 342 to validate a known dataset size; None disables check

# Optional filename-based city/region labels for summary + ANOVA. Empty = 'Unknown'.
# Example: {'Region_A': ['prefix_a', 'prefix_b'], 'Region_B': ['prefix_c']}
LOCATION_KEYWORDS = {}

# Metric CAM is fused from these backbone stages, each normalized before averaging.
CAM_LAYER_NAMES = ('layer2', 'layer3', 'layer4')
# =============================================================================


_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def choose_device():
    if DEVICE_REQUEST == 'auto':
        return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    device = torch.device(DEVICE_REQUEST)
    if device.type == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError(f'Device {DEVICE_REQUEST!r} was requested, but CUDA is unavailable.')
    return device


DEVICE = choose_device()

## 3. Grad-CAM functions and Excel export

This cell defines model loading, fused Grad-CAM calculation, PNG saving, metric aggregation, and the analysis runner.

In [ ]:
def location_from_filename(filename: str) -> str:
    name = filename.lower()
    for location, keywords in LOCATION_KEYWORDS.items():
        if any(keyword.lower() in name for keyword in keywords):
            return location
    return 'Unknown'


def index_source_images():
    """Index image files by basename. If basenames collide, the first source directory wins."""
    image_index = {}
    valid_suffixes = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    for root in SOURCE_IMAGE_DIRS:
        if not root.exists():
            print(f'[WARN] Image directory does not exist: {root}')
            continue
        print(f'[INDEX] {root}')
        for path in root.rglob('*'):
            if path.is_file() and path.suffix.lower() in valid_suffixes and not path.stem.lower().endswith('_mask'):
                if path.name in image_index:
                    warnings.warn(f'Duplicate basename {path.name!r}; keeping {image_index[path.name]} and skipping {path}')
                else:
                    image_index[path.name] = path
    return image_index


def resolve_model_files():
    if not MODEL_DIR.exists():
        raise FileNotFoundError(f'Model directory not found: {MODEL_DIR}')
    if MODEL_FILES is None:
        paths = sorted(MODEL_DIR.glob(MODEL_GLOB))
    else:
        paths = [MODEL_DIR / name for name in MODEL_FILES]
    if not paths:
        raise FileNotFoundError(f'No model checkpoints found in {MODEL_DIR} matching {MODEL_GLOB!r}.')
    absent = [str(path) for path in paths if not path.is_file()]
    if absent:
        raise FileNotFoundError('Missing model checkpoint(s):\n' + '\n'.join(absent))
    return paths


def load_model(checkpoint_path: Path):
    name = checkpoint_path.name.lower()
    if 'resnet101' in name:
        architecture = tv_models.segmentation.deeplabv3_resnet101
    elif 'resnet50' in name:
        architecture = tv_models.segmentation.deeplabv3_resnet50
    else:
        raise ValueError(
            f'Cannot infer architecture from {checkpoint_path.name!r}. '
            'Checkpoint filename must contain "resnet50" or "resnet101", '
            'or update load_model() for your architecture.'
        )

    model = architecture(weights=None, weights_backbone=None, num_classes=NUM_CLASSES, aux_loss=True)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    if isinstance(checkpoint, dict):
        state = checkpoint.get('model_state_dict', checkpoint.get('state_dict', checkpoint))
    else:
        state = checkpoint
    if not isinstance(state, dict):
        raise TypeError(f'Unsupported checkpoint structure in {checkpoint_path}')

    cleaned_state = {}
    for key, value in state.items():
        cleaned_state[key[7:] if key.startswith('module.') else key] = value
    missing, unexpected = model.load_state_dict(cleaned_state, strict=False)
    if missing:
        print(f'[WARN] {checkpoint_path.name}: {len(missing)} missing state keys; examples: {missing[:3]}')
    if unexpected:
        print(f'[WARN] {checkpoint_path.name}: {len(unexpected)} unexpected state keys; examples: {unexpected[:3]}')
    return model.to(DEVICE).eval()


def normalize_cam(cam: torch.Tensor) -> torch.Tensor:
    cam = cam - cam.min()
    maximum = cam.max()
    if maximum.item() > 1e-12:
        cam = cam / maximum
    else:
        cam = torch.zeros_like(cam)
    return cam


def gradcam_and_metrics(model, input_tensor: torch.Tensor, class_id: int):
    """Create fused layer2/3/4 Grad-CAM and metrics for a predicted class.

    Metric mask is the model's predicted class mask, matching the original producer logic.
    Entropy uses the natural logarithm. No map is returned if the class is not predicted.
    """
    activations = {}
    handles = []
    layer_modules = {
        'layer2': model.backbone.layer2,
        'layer3': model.backbone.layer3,
        'layer4': model.backbone.layer4,
    }
    for layer_name in CAM_LAYER_NAMES:
        if layer_name not in layer_modules:
            raise ValueError(f'Unsupported CAM layer in CAM_LAYER_NAMES: {layer_name}')
        handles.append(
            layer_modules[layer_name].register_forward_hook(
                lambda module, inputs, output, name=layer_name: activations.__setitem__(name, output)
            )
        )

    try:
        model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = model(input_tensor)['out']
            logits = F.interpolate(
                logits, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False
            )
            predicted = logits.argmax(dim=1)[0]
            mask = predicted == class_id
            if not mask.any():
                return None
            class_score = (logits[0, class_id] * mask.float()).sum() / (mask.sum().float() + 1e-6)
            tensors = [activations[name] for name in CAM_LAYER_NAMES]
            gradients = torch.autograd.grad(
                class_score, tensors, retain_graph=False, create_graph=False, allow_unused=True
            )

        cams = []
        for activation, gradient in zip(tensors, gradients):
            if gradient is None:
                continue
            channel_weights = gradient.mean(dim=(2, 3), keepdim=True)
            cam = F.relu((channel_weights * activation).sum(dim=1, keepdim=True))
            cam = F.interpolate(cam, size=predicted.shape, mode='bilinear', align_corners=False)[0, 0].detach()
            cams.append(normalize_cam(cam))
        if not cams:
            return None

        fused_cam = normalize_cam(torch.stack(cams).mean(dim=0))
        mask_float = mask.float()
        inside_sum = (fused_cam * mask_float).sum()
        total_sum = fused_cam.sum() + 1e-8
        focus_ratio = (inside_sum / total_sum).item()
        inside_values = fused_cam[mask]
        outside_values = fused_cam[~mask]
        mean_inside = inside_values.mean().item() if inside_values.numel() else 0.0
        mean_outside = outside_values.mean().item() if outside_values.numel() else 0.0

        probabilities = fused_cam.flatten()
        probabilities = probabilities / (probabilities.sum() + 1e-8)
        entropy = (-(probabilities * torch.log(probabilities + 1e-8)).sum()).item()

        height, width = fused_cam.shape
        y_grid, x_grid = torch.meshgrid(
            torch.arange(height, device=DEVICE, dtype=fused_cam.dtype),
            torch.arange(width, device=DEVICE, dtype=fused_cam.dtype),
            indexing='ij',
        )
        if inside_values.numel() and inside_sum.item() > 1e-8:
            cam_y = (y_grid * fused_cam * mask_float).sum() / (inside_sum + 1e-8)
            cam_x = (x_grid * fused_cam * mask_float).sum() / (inside_sum + 1e-8)
            mask_y, mask_x = torch.where(mask)
            mask_center_y, mask_center_x = mask_y.float().mean(), mask_x.float().mean()
            centroid_distance = torch.sqrt(
                (cam_y - mask_center_y) ** 2 + (cam_x - mask_center_x) ** 2
            ).item()
        else:
            centroid_distance = 0.0

        metrics = {
            'Focus Ratio': focus_ratio,
            'Mean CAM Inside': mean_inside,
            'Mean CAM Outside': mean_outside,
            'Entropy': entropy,
            'Centroid Distance': centroid_distance,
        }
        return fused_cam.detach(), predicted.detach(), metrics
    finally:
        for handle in handles:
            handle.remove()
        model.zero_grad(set_to_none=True)


def save_visuals(image_rgb: Image.Image, cam: torch.Tensor, model_name: str, filename: str, class_name: str):
    """Save colour heatmap and image overlay as PNG files."""
    import numpy as np
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.cm as cm

    cam_array = cam.float().cpu().numpy().clip(0, 1)
    colour_map = cm.get_cmap(HEATMAP_COLORMAP)
    heat_rgb = (colour_map(cam_array)[..., :3] * 255).astype('uint8')
    stem = Path(filename).stem
    output_folder = OUTPUT_DIR / 'png' / model_name / class_name
    output_folder.mkdir(parents=True, exist_ok=True)
    Image.fromarray(heat_rgb, mode='RGB').save(output_folder / f'{stem}__heatmap.png')

    original_array = np.asarray(image_rgb.resize((cam_array.shape[1], cam_array.shape[0]), Image.Resampling.BILINEAR), dtype='float32')
    overlay_array = (
        original_array * (1 - OVERLAY_ALPHA) + heat_rgb.astype('float32') * OVERLAY_ALPHA
    ).clip(0, 255).astype('uint8')
    Image.fromarray(overlay_array, mode='RGB').save(output_folder / f'{stem}__overlay.png')


def create_excel(rows, errors):
    from scipy import stats

    raw = pd.DataFrame(rows)
    summary = raw.groupby(['Location', 'Class']).agg({
        'Focus Ratio': ['mean', 'std'],
        'Mean CAM Inside': ['mean', 'std'],
        'Mean CAM Outside': ['mean', 'std'],
        'Entropy': ['mean', 'std'],
        'Centroid Distance': ['mean', 'std'],
        'Image': 'count',
    }).reset_index()

    anova_rows = []
    for class_name in CLASS_NAMES.values():
        subset = raw[raw['Class'] == class_name]
        for metric in ['Focus Ratio', 'Mean CAM Inside', 'Mean CAM Outside', 'Entropy', 'Centroid Distance']:
            groups = [group[metric].dropna().values for _, group in subset.groupby('Location') if len(group[metric].dropna()) > 2]
            if len(groups) > 1:
                f_value, p_value = stats.f_oneway(*groups)
                anova_rows.append({
                    'Class': class_name,
                    'Metric': metric,
                    'F-Statistic': f_value,
                    'p-value': p_value,
                    'Significant? (p<0.05)': 'Yes' if p_value < 0.05 else 'No',
                })
    anova = pd.DataFrame(anova_rows)
    output_xlsx = OUTPUT_DIR / 'gradcam_metrics.xlsx'
    with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:
        raw.to_excel(writer, sheet_name='Raw_Data', index=False)
        summary.to_excel(writer, sheet_name='Location_Class_Summary', index=False)
        anova.to_excel(writer, sheet_name='ANOVA_Analysis', index=False)
        if errors:
            pd.DataFrame(errors).to_excel(writer, sheet_name='Errors', index=False)
    return output_xlsx, len(raw)


def run_analysis():
    if NUM_CLASSES < 2:
        raise ValueError('NUM_CLASSES must include background class 0 and at least one foreground class.')
    invalid_class_ids = [class_id for class_id in CLASS_NAMES if class_id <= 0 or class_id >= NUM_CLASSES]
    if invalid_class_ids:
        raise ValueError(f'CLASS_NAMES has class IDs outside 1..{NUM_CLASSES - 1}: {invalid_class_ids}')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if not IMAGE_LIST_FILE.is_file():
        raise FileNotFoundError(f'Image list file not found: {IMAGE_LIST_FILE}')
    filenames = list(dict.fromkeys(
        line.strip() for line in IMAGE_LIST_FILE.read_text(encoding='utf-8').splitlines() if line.strip()
    ))
    print(f'[INFO] Unique filenames in list: {len(filenames)}')
    if EXPECTED_IMAGE_COUNT is not None and len(filenames) != EXPECTED_IMAGE_COUNT:
        warnings.warn(f'Expected {EXPECTED_IMAGE_COUNT} filenames, found {len(filenames)}.')

    image_index = index_source_images()
    resolved = [(name, image_index.get(name)) for name in filenames]
    missing = [name for name, path in resolved if path is None]
    if missing:
        (OUTPUT_DIR / 'missing_images.txt').write_text('\n'.join(missing), encoding='utf-8')
        print(f'[WARN] {len(missing)} images were not found; see {OUTPUT_DIR / "missing_images.txt"}')

    model_paths = resolve_model_files()
    rows, errors = [], []
    print(f'[INFO] Device: {DEVICE}; models: {len(model_paths)}; resolved images: {len(filenames) - len(missing)}')

    for model_path in model_paths:
        model_name = model_path.stem
        print(f'\n[MODEL] {model_name}')
        model = load_model(model_path)
        try:
            for index, (filename, image_path) in enumerate(resolved, start=1):
                if image_path is None:
                    continue
                try:
                    original = Image.open(image_path).convert('RGB')
                    processed_image = original.resize(INPUT_SIZE, Image.Resampling.BILINEAR) if INPUT_SIZE else original
                    # PIL -> tensor without torchvision's NumPy-backed ToTensor helper.
                    image_bytes = torch.frombuffer(bytearray(processed_image.tobytes()), dtype=torch.uint8)
                    tensor = image_bytes.reshape(processed_image.height, processed_image.width, 3)
                    tensor = tensor.permute(2, 0, 1).float().div_(255).unsqueeze(0)
                    tensor = ((tensor - _MEAN) / _STD).to(DEVICE)
                    location = location_from_filename(filename)

                    for class_id, class_name in CLASS_NAMES.items():
                        result = gradcam_and_metrics(model, tensor, class_id)
                        if result is None:
                            continue
                        cam, _predicted, metrics = result
                        save_visuals(processed_image, cam, model_name, filename, class_name)
                        rows.append({
                            'Model': model_path.name,
                            'Image': filename,
                            'Location': location,
                            'Class ID': class_id,
                            'Class': class_name,
                            **metrics,
                        })
                    if index % 25 == 0 or index == len(resolved):
                        print(f'  {index}/{len(resolved)} images scanned; metric rows={len(rows)}')
                except Exception as exc:
                    errors.append({'Model': model_name, 'Image': filename, 'Error': repr(exc)})
                    print(f'[ERROR] {model_name} / {filename}: {exc}')
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not rows:
        raise RuntimeError('No metrics were generated. Check paths, labels, checkpoint architecture and class count.')
    output_file, row_count = create_excel(rows, errors)
    print(f'\n[DONE] Excel workbook: {output_file}')
    print(f'[DONE] PNG output: {OUTPUT_DIR / "png"}')
    print(f'[DONE] Metric rows: {row_count}; recorded errors: {len(errors)}')
    print('[INFO] A predicted class only gets visualizations when that class is present in the predicted mask.')

## 4. Run analysis

The run can take a while depending on the number and size of images, checkpoints, and available hardware.

In [ ]:
run_analysis()